# Concatenating the aligned, cropped legs — cubes *and* labels

`Code/concat.ipynb` concatenated the legs as they came off the instrument. But the
seven excitation cubes of a leg are **not pixel-aligned** — the animal moves between
exposures — so on five of the eight legs a "spectrum" read down those 217 channels
crosses different tissue at the 31-channel block boundaries.

`Code/Anotation_tool/` fixed that: every leg was registered, shifted by whole pixels
and cropped to one rectangle, then hand-labelled. This notebook concatenates *those*
cubes, the same four ways, and carries the labels across with them.

| tree | channels | `Distinct_cubes/` |
|---|---|---|
| `Alligned_Cropped_All_channels_kept/` | 217 | the aligned export, as it is |
| `Alligned_cropped_overexposed_channels_removed/` | 210 | the same, minus the bands that clip at 4040 counts |

**Four ways** — the pixels are identical in all of them, only the scaling differs:

| way | what it does | scopes |
|---|---|---|
| 1 | raw counts, nothing rescaled | – |
| 2 | z-norm of the 1–99% values, **per channel** | one per channel |
| 3 | z-norm of the 1–99% values, **per excitation cube** | one per 31-band block |
| 4 | z-norm of the 1–99% values, **over the whole 4D cube** | one |

**Labels.** After the crop every cube of a leg shares one grid, so the seven per-cube
masks the annotator wrote are byte-identical — the concatenated cube's mask is simply
that one image. It lands in `Concated_cubes_labels/<leg>_concat.png`: a palette PNG
whose pixel values are class ids, `(Y, X)`, exactly the cube's frame, beside the
`classes.json` that names them and a `labels.json` index.

The machinery is not repeated here. It lives in `Code/concat_lib.py`, extracted
verbatim from `concat.ipynb` so the two cannot drift. Run Step 0 and Step 1, then
Part A or Part B.


## Step 0 — the trees


In [1]:
# =============================================================================
# Step 0: where everything lives.
#
# Two trees, each already holding the aligned Distinct_cubes/ — and, for the
# 217-channel one, the labels the annotator painted. This notebook fills in
# everything else:
#
#   both trees   Concated_cubes/<way>/        the four concatenations
#                Concated_cubes_labels/       one mask per leg, beside them
#   trimmed tree Distinct_cubes/              the per-cube cubes, trimmed
#                Distinct_cubes_labels/       the same masks, copied across
#
# The all-channels tree is read from and never written to outside its two
# Concated_ folders, and nothing under Data/Raw is opened at all.
# =============================================================================
import json
import shutil
import sys
from pathlib import Path

import numpy as np
from PIL import Image

# concat_lib sits next to this notebook; find it whether the kernel started in
# Code/ or in the project root.
HERE = Path.cwd()
CODE = next(p for p in (HERE, HERE / "Code", HERE.parent) if (p / "concat_lib.py").exists())
sys.path.insert(0, str(CODE))
import concat_lib as L

L.BUILDER = "Code/concat_aligned.ipynb"        # stamped into every cube's UserComments

PROCESSED = L.PROCESSED_ROOT
ALL_KEPT = PROCESSED / "Alligned_Cropped_All_channels_kept"
TRIMMED = PROCESSED / "Alligned_cropped_overexposed_channels_removed"

TRIM = {"385": 2, "400": 5}                    # leading channels to drop, by excitation
IGNORE = 255                                   # the annotator's "not labelled" value
SATURATED = 4040                               # the sensor's ceiling


def distinct(tree):        return tree / "Distinct_cubes"
def distinct_labels(tree): return tree / "Distinct_cubes_labels"
def concat_root(tree):     return tree / "Concated_cubes"
def concat_raw(tree):      return concat_root(tree) / "Raw_Concatenated"
def concat_labels(tree):   return tree / "Concated_cubes_labels"


for tree in (ALL_KEPT, TRIMMED):
    print(f"{tree.name}")
    for sub in (distinct, distinct_labels, concat_root, concat_labels):
        d = sub(tree)
        n = len(list(d.iterdir())) if d.exists() else None
        print(f"    {d.name:24s} {'missing' if n is None else f'{n} entries'}")


Alligned_Cropped_All_channels_kept
    Distinct_cubes           6 entries
    Distinct_cubes_labels    7 entries
    Concated_cubes           4 entries
    Concated_cubes_labels    0 entries
Alligned_cropped_overexposed_channels_removed
    Distinct_cubes           0 entries
    Distinct_cubes_labels    missing
    Concated_cubes           0 entries
    Concated_cubes_labels    missing


## Step 1 — the legs, and the one mask each of them carries


In [2]:
# =============================================================================
# Step 1: the legs, and the one mask each of them carries.
#
# The annotator paints ONE mask per leg, in the reference cube's frame, and
# saves it warped into every cube's own frame. Before the crop those seven PNGs
# genuinely differ — a pixel that falls off a shifted cube becomes 255. After
# the crop they cannot differ: the box is inside the region every cube sees, so
# all seven frames ARE the same frame and the seven files come out
# byte-identical. That identity is the whole reason a single mask for a
# 217-channel cube means anything, so it is checked rather than assumed.
# =============================================================================
def read_leg_label(label_dir):
    """The leg's single mask, with a check that all its cubes agree on it.

    The PNGs are palette images whose indices are the class ids (the palette is
    only so they are viewable), and numpy reads a "P" image as those indices.
    """
    label_dir = Path(label_dir)
    pngs = sorted(label_dir.glob("*.png"))
    if not pngs:
        raise FileNotFoundError(f"no label PNGs in {label_dir}")
    masks = [np.array(Image.open(p)) for p in pngs]
    first = masks[0]
    differ = [p.name for p, m in zip(pngs, masks)
              if m.shape != first.shape or not np.array_equal(m, first)]
    if differ:
        raise ValueError(f"{label_dir.name}: the per-cube masks are not identical "
                         f"({differ}) — these cubes are not on a common grid, so one "
                         "mask cannot describe their concatenation")
    return first, pngs


def class_counts(mask, labels):
    """Pixels per class, named — plus anything in the mask that isn't a class."""
    out = {}
    for i, n in zip(*np.unique(mask, return_counts=True)):
        i = int(i)
        name = labels.get(str(i)) or ("unlabelled" if i == IGNORE else f"id {i}")
        out[name] = int(n)
    return out


CLASSES = json.loads((distinct_labels(ALL_KEPT) / "classes.json").read_text())
LABELS = CLASSES["labels"]

legs = L.find_legs_flat(distinct(ALL_KEPT))
print(f"{len(legs)} legs under {distinct(ALL_KEPT)}")
print(f"classes: " + ", ".join(f"{k}={v}" for k, v in LABELS.items()) + f"  ({IGNORE}=unlabelled)\n")

for leg in legs:
    name = L.leg_name(leg)
    infos = [L.read_cube_info(p) for p in leg["cubes"]]
    frames = {(i["Y"], i["X"]) for i in infos}
    if len(frames) != 1:
        raise ValueError(f"{name}: cubes disagree on frame size: {frames}")
    Y, X = frames.pop()
    mask, pngs = read_leg_label(distinct_labels(ALL_KEPT) / name)
    if mask.shape != (Y, X):
        raise ValueError(f"{name}: mask is {mask.shape}, cubes are {(Y, X)}")
    counts = class_counts(mask, LABELS)
    print(f"{name:24s} {len(infos)} cubes x {infos[0]['C']:3d} ch  {Y:4d}x{X:<4d}  "
          f"mask from {len(pngs)} identical PNGs  "
          + "  ".join(f"{k} {v / mask.size:5.1%}" for k, v in counts.items()))


6 legs under /Users/zmkoyan/User Files/multi-excitation-hsi-in-vivo-rats/Data/Processed/Alligned_Cropped_All_channels_kept/Distinct_cubes
classes: 1=background, 2=vessel, 3=nerve, 4=muscle  (255=unlabelled)

12.09_Rat_1_Left_leg     7 cubes x  31 ch   161x241   mask from 7 identical PNGs  vessel 21.3%  nerve 10.6%  muscle 68.1%
12.09_Rat_1_Right_leg    7 cubes x  31 ch   229x218   mask from 7 identical PNGs  vessel 16.5%  nerve 10.7%  muscle 72.9%
12.09_Rat_2_Left_leg     7 cubes x  31 ch   307x406   mask from 7 identical PNGs  vessel 24.2%  nerve 10.6%  muscle 65.1%
12.09_Rat_2_Right_leg    7 cubes x  31 ch   452x514   mask from 7 identical PNGs  vessel 12.9%  nerve  5.7%  muscle 81.4%
12.09_Rat_3_Left_leg     7 cubes x  31 ch   211x323   mask from 7 identical PNGs  vessel 13.6%  nerve 11.8%  muscle 74.6%
12.09_Rat_3_Right_leg    7 cubes x  31 ch   199x285   mask from 7 identical PNGs  vessel 13.3%  nerve 14.1%  muscle 72.6%


# Part A — all channels kept (217 channels)

→ `Data/Processed/Alligned_Cropped_All_channels_kept/`

Every channel of every cube: 7 × 31, in excitation order 310 / 325 / 340 / 365 / 385 /
400 with the white-light cube last. Channel `31k + i` is band `i` of the `k`-th cube.


## Way 1 — raw concatenation


In [3]:
# =============================================================================
# Part A, way 1 — the raw concatenation.
#
# Identical in every respect to Part 1 of concat.ipynb; only the source cubes
# differ, being the aligned and cropped ones rather than the raw ones. The
# pixels are copied through untouched, so the check below is exact: every
# sampled pixel of the result must equal the source channel it came from.
#
# Output: Alligned_Cropped_All_channels_kept/Concated_cubes/Raw_Concatenated
# =============================================================================
L.build_way1(concat_raw(ALL_KEPT), legs)
L.verify_way1(concat_raw(ALL_KEPT))


6 legs  ->  /Users/zmkoyan/User Files/multi-excitation-hsi-in-vivo-rats/Data/Processed/Alligned_Cropped_All_channels_kept/Concated_cubes/Raw_Concatenated

12.09_Rat_1_Left_leg_concat.im3    exists, skipped
12.09_Rat_1_Right_leg_concat.im3   exists, skipped
12.09_Rat_2_Left_leg_concat.im3    exists, skipped
12.09_Rat_2_Right_leg_concat.im3   exists, skipped
12.09_Rat_3_Left_leg_concat.im3    exists, skipped
12.09_Rat_3_Right_leg_concat.im3   exists, skipped
12.09_Rat_1_Left_leg_concat.im3  Shape=(241, 161, 217)  Data=217 channels  bands in Spectra=217  emission 420-720 nm
    block 0  ch   0- 30  310_20000exp.im3     OK
    block 1  ch  31- 61  325_5000exp.im3      OK
    block 2  ch  62- 92  340_5000exp.im3      OK
    block 3  ch  93-123  365_3000exp.im3      OK
    block 4  ch 124-154  385_1500exp.im3      OK
    block 5  ch 155-185  400_1500exp.im3      OK
    block 6  ch 186-216  white_4exp.im3       OK
12.09_Rat_1_Right_leg_concat.im3  Shape=(218, 229, 217)  Data=217 channels  ban

## Way 2 — z-norm per channel


In [4]:
# =============================================================================
# Part A, way 2 — z-norm PER CHANNEL.
#
# 217 independent scopes: each channel is centred and scaled by its own 1-99%
# mean and std. Every channel ends up with comparable contrast, which is what
# makes faint bands readable, but it also erases the brightness relationship
# BETWEEN bands — a dim channel is stretched exactly as hard as a bright one,
# so a spectrum read across channels is no longer the shape the tissue emitted.
# Use it for looking at one channel at a time, not for spectral decomposition.
#
# Output: .../Concated_cubes/Per_Chanell_Norm_1_99_Concat
# =============================================================================
L.run_way("per_channel", concat_raw(ALL_KEPT), concat_root(ALL_KEPT))


way 2 - z-norm per channel  ->  /Users/zmkoyan/User Files/multi-excitation-hsi-in-vivo-rats/Data/Processed/Alligned_Cropped_All_channels_kept/Concated_cubes/Per_Chanell_Norm_1_99_Concat

  12.09_Rat_1_Left_leg_concat_znorm_per_channel.im3    exists, skipped
  12.09_Rat_1_Right_leg_concat_znorm_per_channel.im3   exists, skipped
  12.09_Rat_2_Left_leg_concat_znorm_per_channel.im3    exists, skipped
  12.09_Rat_2_Right_leg_concat_znorm_per_channel.im3   exists, skipped
  12.09_Rat_3_Left_leg_concat_znorm_per_channel.im3    exists, skipped
  12.09_Rat_3_Right_leg_concat_znorm_per_channel.im3   exists, skipped


## Way 3 — z-norm per excitation cube


In [5]:
# =============================================================================
# Part A, way 3 — z-norm PER EXCITATION CUBE.
#
# 7 scopes, one per 31-channel block: all 31 emission bands of an excitation
# share a single mean and std. Within a block the emission spectrum keeps its
# true shape (a dim band stays dim), while the huge exposure differences BETWEEN
# excitations — 20000 ms at 310 nm against 4 ms for white light — are divided
# out. This is usually the one to reach for when comparing excitations.
#
# Output: .../Concated_cubes/Per_Cube_Norm_1_99_Concat
# =============================================================================
L.run_way("per_excitation", concat_raw(ALL_KEPT), concat_root(ALL_KEPT))


way 3 - z-norm per excitation cube  ->  /Users/zmkoyan/User Files/multi-excitation-hsi-in-vivo-rats/Data/Processed/Alligned_Cropped_All_channels_kept/Concated_cubes/Per_Cube_Norm_1_99_Concat

  12.09_Rat_1_Left_leg_concat_znorm_per_excitation.im3 exists, skipped
  12.09_Rat_1_Right_leg_concat_znorm_per_excitation.im3 exists, skipped
  12.09_Rat_2_Left_leg_concat_znorm_per_excitation.im3 exists, skipped
  12.09_Rat_2_Right_leg_concat_znorm_per_excitation.im3 exists, skipped
  12.09_Rat_3_Left_leg_concat_znorm_per_excitation.im3 exists, skipped
  12.09_Rat_3_Right_leg_concat_znorm_per_excitation.im3 exists, skipped


## Way 4 — z-norm over the whole 4D cube


In [6]:
# =============================================================================
# Part A, way 4 — z-norm over the WHOLE 4D CUBE.
#
# One scope: a single mean and std for all 217 channels of the leg. Every
# relationship inside the cube is preserved exactly — between bands and between
# excitations alike — because this is one affine map applied to everything. The
# most faithful of the three and the least corrective: the 310 nm block stays
# brighter than the white block if that is how it was acquired.
#
# Output: .../Concated_cubes/Overall_4D_Norm_1_99_Concat
# =============================================================================
L.run_way("whole_cube", concat_raw(ALL_KEPT), concat_root(ALL_KEPT))


way 4 - z-norm over the whole 4D cube  ->  /Users/zmkoyan/User Files/multi-excitation-hsi-in-vivo-rats/Data/Processed/Alligned_Cropped_All_channels_kept/Concated_cubes/Overall_4D_Norm_1_99_Concat

  12.09_Rat_1_Left_leg_concat_znorm_whole_cube.im3     exists, skipped
  12.09_Rat_1_Right_leg_concat_znorm_whole_cube.im3    exists, skipped
  12.09_Rat_2_Left_leg_concat_znorm_whole_cube.im3     exists, skipped
  12.09_Rat_2_Right_leg_concat_znorm_whole_cube.im3    exists, skipped
  12.09_Rat_3_Left_leg_concat_znorm_whole_cube.im3     exists, skipped
  12.09_Rat_3_Right_leg_concat_znorm_whole_cube.im3    exists, skipped


### Check — ways 2–4


In [7]:
# =============================================================================
# Part A, check: invert a sample of pixels of every normalised cube back
# through its own recorded transform and compare with the raw cube it came
# from. The round trip is not bit-exact — z is quantised into 65535 steps on
# the way in — so what is reported is the worst error in counts, next to the
# step size in sigma that produced it.
# =============================================================================
L.verify_norm(concat_root(ALL_KEPT))


way 2 - z-norm per channel
  12.09_Rat_1_Left_leg_concat_znorm_per_channel.im3    C=217 scopes=217  z -6.65..+75.28  step 0.0013 sigma  round trip <= 0.62 counts  u 598-27257
  12.09_Rat_1_Right_leg_concat_znorm_per_channel.im3   C=217 scopes=217  z -1045.00..+29.20  step 0.0164 sigma  round trip <= 6.08 counts  u 7260-64271
  12.09_Rat_2_Left_leg_concat_znorm_per_channel.im3    C=217 scopes=217  z -200.51..+22.00  step 0.0034 sigma  round trip <= 1.76 counts  u 10577-61044
  12.09_Rat_2_Right_leg_concat_znorm_per_channel.im3   C=217 scopes=217  z -16.21..+29.64  step 0.0007 sigma  round trip <= 0.30 counts  u 7948-36749
  12.09_Rat_3_Left_leg_concat_znorm_per_channel.im3    C=217 scopes=217  z -471.00..+43.30  step 0.0078 sigma  round trip <= 4.60 counts  u 29180-60957
  12.09_Rat_3_Right_leg_concat_znorm_per_channel.im3   C=217 scopes=217  z -13.93..+49.68  step 0.0010 sigma  round trip <= 0.50 counts  u 968-30535

way 3 - z-norm per excitation cube
  12.09_Rat_1_Left_leg_concat_znor

## The labels for the concatenated cubes


In [8]:
# =============================================================================
# Part A, labels: one mask per leg, beside the concatenated cubes.
#
# Nothing has to be done to the mask itself — the concatenation only grows the
# channel axis, and the mask has no channel axis — so the file is COPIED rather
# than re-encoded. That keeps the annotator's palette (which is what makes the
# class ids visible as colours) exactly as it wrote it; re-saving through PIL
# would preserve the indices but is a chance to lose the palette for nothing.
#
# The same mask serves all four ways, so it is written once per tree, not once
# per way. labels.json records, per leg, which per-cube files it stands for and
# how many pixels each class holds.
#
# Output: .../Concated_cubes_labels/<leg>_concat.png + classes.json + labels.json
# =============================================================================
def build_concat_labels(tree, legs, overwrite=False):
    """The per-leg mask, placed next to that leg's concatenated cubes."""
    out_dir = concat_labels(tree)
    out_dir.mkdir(parents=True, exist_ok=True)
    classes = json.loads((distinct_labels(tree) / "classes.json").read_text())
    shutil.copy2(distinct_labels(tree) / "classes.json", out_dir / "classes.json")

    index = {"labelled_by": "Code/Anotation_tool/annotate.py",
             "collected_by": "Code/concat_aligned.ipynb",
             "applies_to": f"every cube under {concat_root(tree).name}/, all four ways",
             "format": "palette PNG, uint8 (Y, X), pixel value = class id",
             "ignore": IGNORE, "labels": classes["labels"],
             "classes": classes["classes"], "legs": []}

    for leg in legs:
        name = L.leg_name(leg)
        mask, pngs = read_leg_label(distinct_labels(tree) / name)
        infos = [L.read_cube_info(c) for c in leg["cubes"]]
        info = infos[0]
        if mask.shape != (info["Y"], info["X"]):
            raise ValueError(f"{name}: mask {mask.shape} != cube {(info['Y'], info['X'])}")

        out_path = out_dir / f"{name}_concat.png"
        if out_path.exists() and not overwrite:
            note = "exists, skipped"
        else:
            shutil.copy2(pngs[0], out_path)
            note = f"from {pngs[0].name}"

        counts = class_counts(mask, classes["labels"])
        index["legs"].append({"leg": name, "png": out_path.name,
                              "cube": f"{name}_concat.im3",
                              "C": sum(i["C"] for i in infos),
                              "Y": int(mask.shape[0]), "X": int(mask.shape[1]),
                              "identical_per_cube_masks": [p.name for p in pngs],
                              "pixels": counts})
        print(f"  {out_path.name:34s} {mask.shape[0]:4d}x{mask.shape[1]:<4d}  "
              + "  ".join(f"{k} {v}" for k, v in counts.items()) + f"   ({note})")

    (out_dir / "labels.json").write_text(json.dumps(index, indent=1))
    print(f"\n  + classes.json, labels.json  ->  {out_dir}")
    return out_dir


build_concat_labels(ALL_KEPT, legs)


  12.09_Rat_1_Left_leg_concat.png     161x241   vessel 8274  nerve 4120  muscle 26407   (from 310_20000exp.png)
  12.09_Rat_1_Right_leg_concat.png    229x218   vessel 8217  nerve 5319  muscle 36386   (from 310_20000exp.png)
  12.09_Rat_2_Left_leg_concat.png     307x406   vessel 30195  nerve 13253  muscle 81194   (from 310_20000exp.png)
  12.09_Rat_2_Right_leg_concat.png    452x514   vessel 29893  nerve 13291  muscle 189144   (from 310_20000exp.png)
  12.09_Rat_3_Left_leg_concat.png     211x323   vessel 9289  nerve 8027  muscle 50837   (from 310_20000exp.png)
  12.09_Rat_3_Right_leg_concat.png    199x285   vessel 7564  nerve 7987  muscle 41164   (from 310_17000exp.png)

  + classes.json, labels.json  ->  /Users/zmkoyan/User Files/multi-excitation-hsi-in-vivo-rats/Data/Processed/Alligned_Cropped_All_channels_kept/Concated_cubes_labels


# Part B — overexposed channels removed (210 channels)

→ `Data/Processed/Alligned_cropped_overexposed_channels_removed/`

The 385 nm cube loses its first 2 channels and the 400 nm cube its first 5 — the bands
that clip at 4040 counts in every leg. They are removed from the pixels *and* the band
records, so no statistic in ways 2–4 ever sees them.

Block layout after the trim: `310 [0:31] 325 [31:62] 340 [62:93] 365 [93:124]
385 [124:153] 400 [153:179] white [179:210]`.

This tree starts empty, so unlike Part A it needs its `Distinct_cubes/` built first.


## Step B1 — the trimmed distinct cubes


In [9]:
# =============================================================================
# Part B, step 1: the trimmed per-cube cubes.
#
# The aligned export exists only in the all-channels tree, so this tree is
# derived from it rather than re-exported — the alignment and the crop are
# already settled and re-running them could only introduce a difference. Each
# cube is rewritten with its overexposed leading bands dropped from the pixels
# AND from the Spectra records, which is exactly what write_concat does to a
# whole leg, applied to a manifest holding a single cube:
#
#     385 nm cube  ->  first 2 channels dropped  (31 -> 29)
#     400 nm cube  ->  first 5 channels dropped  (31 -> 26)
#
# The trim is the one concat.ipynb settled on, and it still fits these six legs:
# the first 400 nm band is 95-100% clipped in every one of them and the first
# 385 nm band 3-42%, while nothing that survives exceeds ~2% (printed below).
# Keeping the same numbers also keeps this tree channel-comparable with
# Data/Processed/Overexposed_channels_removed.
# =============================================================================
def build_trimmed_distinct(src_tree, out_tree, legs, trim, overwrite=False):
    """Copy every leg's cubes into out_tree with the trimmed channels removed."""
    print(f"{len(legs)} legs  ->  {distinct(out_tree)}\n")
    for leg in legs:
        name = L.leg_name(leg)
        out_dir = distinct(out_tree) / name
        out_dir.mkdir(parents=True, exist_ok=True)
        rows, total_kept = [], 0

        for cube in leg["cubes"]:
            # A one-cube leg: build_manifest applies the trim and slices the
            # band records, write_concat writes a perfectly ordinary .im3.
            man = L.build_manifest({**leg, "cubes": [cube]}, trim)
            info = man["infos"][0]
            out_path = out_dir / cube.name
            if out_path.exists() and not overwrite:
                note = "exists, skipped"
            else:
                _, size = L.write_concat(man, out_path)
                note = f"{size / 1e6:.1f} MB"
            dropped = f"dropped {info['start']}" if info["start"] else "kept whole"
            rows.append({"file": cube.name, "excitation": info["excitation"],
                         "exposure_ms": info["exposure_ms_filename"],
                         "source_channels": [info["start"], info["stop"] - 1],
                         "dropped_leading": info["start"], "channels": man["C"]})
            total_kept += man["C"]
            print(f"  {name:24s} {cube.name:20s} {info['C']:3d} -> {man['C']:3d} ch  "
                  f"{dropped:12s} {note}")

        (out_dir / "_trim.json").write_text(json.dumps(
            {"leg": name, "source": str(distinct(src_tree) / name),
             "trimmed_by": "Code/concat_aligned.ipynb",
             "reason": f"leading bands clipping at {SATURATED} counts",
             "trim_leading_by_excitation": trim,
             "channels_total": total_kept, "cubes": rows}, indent=1))
        print(f"  {'':24s} {'_trim.json':20s} {total_kept} channels across {len(rows)} cubes\n")


build_trimmed_distinct(ALL_KEPT, TRIMMED, legs, TRIM)


6 legs  ->  /Users/zmkoyan/User Files/multi-excitation-hsi-in-vivo-rats/Data/Processed/Alligned_cropped_overexposed_channels_removed/Distinct_cubes

  12.09_Rat_1_Left_leg     310_20000exp.im3      31 ->  31 ch  kept whole   2.6 MB
  12.09_Rat_1_Left_leg     325_5000exp.im3       31 ->  31 ch  kept whole   2.6 MB
  12.09_Rat_1_Left_leg     340_5000exp.im3       31 ->  31 ch  kept whole   2.6 MB
  12.09_Rat_1_Left_leg     365_3000exp.im3       31 ->  31 ch  kept whole   2.6 MB
  12.09_Rat_1_Left_leg     385_1500exp.im3       31 ->  29 ch  dropped 2    2.5 MB
  12.09_Rat_1_Left_leg     400_1500exp.im3       31 ->  26 ch  dropped 5    2.2 MB
  12.09_Rat_1_Left_leg     white_4exp.im3        31 ->  31 ch  kept whole   2.6 MB
                           _trim.json           210 channels across 7 cubes

  12.09_Rat_1_Right_leg    310_20000exp.im3      31 ->  31 ch  kept whole   3.3 MB
  12.09_Rat_1_Right_leg    325_20000exp.im3      31 ->  31 ch  kept whole   3.3 MB
  12.09_Rat_1_Right_leg    

### Check — the trimmed cubes


In [10]:
# =============================================================================
# Part B, check: every trimmed cube against the cube it came from.
#
# Three things have to hold. The surviving pixels must be the source's last
# C channels, exactly — nothing was rescaled. The Spectra records must have
# been sliced with them, or the file carries 31 band entries for 26 channels.
# And the point of the exercise: what clipping is left.
# =============================================================================
SAMPLE = 2000


def verify_trimmed_distinct(src_tree, out_tree, legs, seed=0):
    rng = np.random.default_rng(seed)
    for leg in legs:
        name = L.leg_name(leg)
        worst_before = worst_after = 0.0
        for cube in leg["cubes"]:
            src_info = L.read_cube_info(cube)
            out_path = distinct(out_tree) / name / cube.name
            out_info = L.read_cube_info(out_path)
            start = src_info["C"] - out_info["C"]

            src = np.memmap(cube, dtype="<u2", mode="r", offset=src_info["data_offset"],
                            shape=(src_info["Y"], src_info["X"], src_info["C"]))
            dst = np.memmap(out_path, dtype="<u2", mode="r", offset=out_info["data_offset"],
                            shape=(out_info["Y"], out_info["X"], out_info["C"]))
            ys = rng.integers(0, out_info["Y"], SAMPLE)
            xs = rng.integers(0, out_info["X"], SAMPLE)
            same = np.array_equal(dst[ys, xs, :], src[ys, xs, start:])
            bands_ok = len(out_info["emission_nm"]) == out_info["C"]
            emission_ok = out_info["emission_nm"] == src_info["emission_nm"][start:]

            sat = [(np.asarray(src[:, :, c]) >= SATURATED).mean() for c in range(src_info["C"])]
            worst_before = max(worst_before, max(sat[:start], default=0.0))
            worst_after = max(worst_after, max(sat[start:]))
            del src, dst

            flag = "OK" if (same and bands_ok and emission_ok) else "MISMATCH"
            drop = f"  (dropped {start})" if start else ""
            print(f"  {name:24s} {cube.name:20s} {out_info['C']:3d} ch  "
                  f"{len(out_info['emission_nm']):3d} band records  "
                  f"{out_info['emission_nm'][0]:.0f}-{out_info['emission_nm'][-1]:.0f} nm  "
                  f"{flag}{drop}")
        print(f"  {name:24s} worst clipping: {worst_before:6.1%} in the dropped bands, "
              f"{worst_after:6.1%} in the kept ones\n")


verify_trimmed_distinct(ALL_KEPT, TRIMMED, legs)


  12.09_Rat_1_Left_leg     310_20000exp.im3      31 ch   31 band records  420-720 nm  OK
  12.09_Rat_1_Left_leg     325_5000exp.im3       31 ch   31 band records  420-720 nm  OK
  12.09_Rat_1_Left_leg     340_5000exp.im3       31 ch   31 band records  420-720 nm  OK
  12.09_Rat_1_Left_leg     365_3000exp.im3       31 ch   31 band records  420-720 nm  OK
  12.09_Rat_1_Left_leg     385_1500exp.im3       29 ch   29 band records  440-720 nm  OK  (dropped 2)
  12.09_Rat_1_Left_leg     400_1500exp.im3       26 ch   26 band records  470-720 nm  OK  (dropped 5)
  12.09_Rat_1_Left_leg     white_4exp.im3        31 ch   31 band records  420-720 nm  OK
  12.09_Rat_1_Left_leg     worst clipping: 100.0% in the dropped bands,   2.6% in the kept ones

  12.09_Rat_1_Right_leg    310_20000exp.im3      31 ch   31 band records  420-720 nm  OK
  12.09_Rat_1_Right_leg    325_20000exp.im3      31 ch   31 band records  420-720 nm  OK
  12.09_Rat_1_Right_leg    340_10000exp.im3      31 ch   31 band records  42

## Step B2 — the labels, carried across


In [11]:
# =============================================================================
# Part B, step 2: the labels.
#
# A mask describes pixels, not channels, so dropping seven bands cannot change
# it — the trimmed tree gets the all-channels tree's masks verbatim, copied so
# each tree stands on its own rather than pointing at its neighbour. The check
# in read_leg_label runs again on the copies and would catch a bad one.
# =============================================================================
def copy_labels(src_tree, dst_tree, legs, overwrite=False):
    """Mirror Distinct_cubes_labels/ from one tree into another."""
    src_root, dst_root = distinct_labels(src_tree), distinct_labels(dst_tree)
    dst_root.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_root / "classes.json", dst_root / "classes.json")

    for leg in legs:
        name = L.leg_name(leg)
        (dst_root / name).mkdir(exist_ok=True)
        copied = 0
        for png in sorted((src_root / name).glob("*.png")):
            out = dst_root / name / png.name
            if out.exists() and not overwrite:
                continue
            shutil.copy2(png, out)
            copied += 1
        mask, pngs = read_leg_label(dst_root / name)
        print(f"  {name:24s} {len(pngs)} masks {mask.shape[0]}x{mask.shape[1]}  "
              f"({copied} copied, {len(pngs) - copied} already there)")
    print(f"\n  + classes.json  ->  {dst_root}")


copy_labels(ALL_KEPT, TRIMMED, legs)

# The trimmed tree's own legs — same names and frames, 26 or 29 channels where
# the trim bit. Everything below uses these, not Part A's.
legs_trimmed = L.find_legs_flat(distinct(TRIMMED))
print()
build_concat_labels(TRIMMED, legs_trimmed)


  12.09_Rat_1_Left_leg     7 masks 161x241  (7 copied, 0 already there)
  12.09_Rat_1_Right_leg    7 masks 229x218  (7 copied, 0 already there)
  12.09_Rat_2_Left_leg     7 masks 307x406  (7 copied, 0 already there)
  12.09_Rat_2_Right_leg    7 masks 452x514  (7 copied, 0 already there)
  12.09_Rat_3_Left_leg     7 masks 211x323  (7 copied, 0 already there)
  12.09_Rat_3_Right_leg    7 masks 199x285  (7 copied, 0 already there)

  + classes.json  ->  /Users/zmkoyan/User Files/multi-excitation-hsi-in-vivo-rats/Data/Processed/Alligned_cropped_overexposed_channels_removed/Distinct_cubes_labels

  12.09_Rat_1_Left_leg_concat.png     161x241   vessel 8274  nerve 4120  muscle 26407   (from 310_20000exp.png)
  12.09_Rat_1_Right_leg_concat.png    229x218   vessel 8217  nerve 5319  muscle 36386   (from 310_20000exp.png)
  12.09_Rat_2_Left_leg_concat.png     307x406   vessel 30195  nerve 13253  muscle 81194   (from 310_20000exp.png)
  12.09_Rat_2_Right_leg_concat.png    452x514   vessel 29893  n

## Way 1 — raw concatenation, trimmed


In [12]:
# =============================================================================
# Part B, way 1 — the 210-channel raw concatenation.
#
# The trim already happened, cube by cube, in step B1, so there is nothing left
# to drop here: these cubes are stacked whole. The check compares each block
# against its trimmed source, exactly.
#
# Output: .../Alligned_cropped_overexposed_channels_removed/Concated_cubes/Raw_Concatenated
# =============================================================================
L.build_way1(concat_raw(TRIMMED), legs_trimmed)
L.verify_way1(concat_raw(TRIMMED))


6 legs  ->  /Users/zmkoyan/User Files/multi-excitation-hsi-in-vivo-rats/Data/Processed/Alligned_cropped_overexposed_channels_removed/Concated_cubes/Raw_Concatenated

12.09_Rat_1_Left_leg         7 cubes -> 210 channels  241x161
    block 0  ch   0- 30   310 nm   20000 ms  310_20000exp.im3
    block 1  ch  31- 61   325 nm    5000 ms  325_5000exp.im3
    block 2  ch  62- 92   340 nm    5000 ms  340_5000exp.im3
    block 3  ch  93-123   365 nm    3000 ms  365_3000exp.im3
    block 4  ch 124-152   385 nm    1500 ms  385_1500exp.im3
    block 5  ch 153-178   400 nm    1500 ms  400_1500exp.im3
    block 6  ch 179-209    white       4 ms  white_4exp.im3
    wrote 12.09_Rat_1_Left_leg_concat.im3  (18 MB) + 12.09_Rat_1_Left_leg_concat_channels.csv, 12.09_Rat_1_Left_leg_concat_channels.json

12.09_Rat_1_Right_leg        7 cubes -> 210 channels  218x229
    block 0  ch   0- 30   310 nm   20000 ms  310_20000exp.im3
    block 1  ch  31- 61   325 nm   20000 ms  325_20000exp.im3
    block 2  ch  62- 

## Way 2 — z-norm per channel, trimmed


In [13]:
# =============================================================================
# Part B, way 2 — z-norm PER CHANNEL on the 210-channel cubes.
#
# Identical to Part A's way 2 except for the input: the dropped channels
# contribute nothing to any mean, std or percentile, because they are not in
# the file at all.
#
# Output: .../Concated_cubes/Per_Chanell_Norm_1_99_Concat
# =============================================================================
L.run_way("per_channel", concat_raw(TRIMMED), concat_root(TRIMMED))


way 2 - z-norm per channel  ->  /Users/zmkoyan/User Files/multi-excitation-hsi-in-vivo-rats/Data/Processed/Alligned_cropped_overexposed_channels_removed/Concated_cubes/Per_Chanell_Norm_1_99_Concat

  12.09_Rat_1_Left_leg_concat_znorm_per_channel.im3    18 MB  z -4.11 .. +75.28
  12.09_Rat_1_Right_leg_concat_znorm_per_channel.im3   22 MB  z -4.81 .. +29.20
  12.09_Rat_2_Left_leg_concat_znorm_per_channel.im3    54 MB  z -2.96 .. +22.00
  12.09_Rat_2_Right_leg_concat_znorm_per_channel.im3   99 MB  z -4.43 .. +29.64
  12.09_Rat_3_Left_leg_concat_znorm_per_channel.im3    30 MB  z -3.30 .. +43.30
  12.09_Rat_3_Right_leg_concat_znorm_per_channel.im3   25 MB  z -2.58 .. +49.68


## Way 3 — z-norm per excitation cube, trimmed


In [14]:
# =============================================================================
# Part B, way 3 — z-norm PER EXCITATION CUBE on the 210-channel cubes.
#
# The 385 and 400 nm scopes now pool 29 and 26 bands instead of 31, and those
# are the bands the clipped ones were dragging upward — this is the way the
# trim changes most.
#
# Output: .../Concated_cubes/Per_Cube_Norm_1_99_Concat
# =============================================================================
L.run_way("per_excitation", concat_raw(TRIMMED), concat_root(TRIMMED))


way 3 - z-norm per excitation cube  ->  /Users/zmkoyan/User Files/multi-excitation-hsi-in-vivo-rats/Data/Processed/Alligned_cropped_overexposed_channels_removed/Concated_cubes/Per_Cube_Norm_1_99_Concat

  12.09_Rat_1_Left_leg_concat_znorm_per_excitation.im3 18 MB  z -1.91 .. +8.33
  12.09_Rat_1_Right_leg_concat_znorm_per_excitation.im3 22 MB  z -2.73 .. +6.94
  12.09_Rat_2_Left_leg_concat_znorm_per_excitation.im3 54 MB  z -1.97 .. +11.96
  12.09_Rat_2_Right_leg_concat_znorm_per_excitation.im3 99 MB  z -2.67 .. +8.85
  12.09_Rat_3_Left_leg_concat_znorm_per_excitation.im3 30 MB  z -2.20 .. +8.33
  12.09_Rat_3_Right_leg_concat_znorm_per_excitation.im3 25 MB  z -2.03 .. +14.03


## Way 4 — z-norm over the whole 4D cube, trimmed


In [15]:
# =============================================================================
# Part B, way 4 — z-norm over the WHOLE 4D CUBE, on the 210-channel cubes.
#
# Output: .../Concated_cubes/Overall_4D_Norm_1_99_Concat
# =============================================================================
L.run_way("whole_cube", concat_raw(TRIMMED), concat_root(TRIMMED))


way 4 - z-norm over the whole 4D cube  ->  /Users/zmkoyan/User Files/multi-excitation-hsi-in-vivo-rats/Data/Processed/Alligned_cropped_overexposed_channels_removed/Concated_cubes/Overall_4D_Norm_1_99_Concat

  12.09_Rat_1_Left_leg_concat_znorm_whole_cube.im3     18 MB  z -1.35 .. +5.09
  12.09_Rat_1_Right_leg_concat_znorm_whole_cube.im3    22 MB  z -1.48 .. +5.72
  12.09_Rat_2_Left_leg_concat_znorm_whole_cube.im3     54 MB  z -1.30 .. +7.17
  12.09_Rat_2_Right_leg_concat_znorm_whole_cube.im3    99 MB  z -1.57 .. +5.75
  12.09_Rat_3_Left_leg_concat_znorm_whole_cube.im3     30 MB  z -1.51 .. +5.38
  12.09_Rat_3_Right_leg_concat_znorm_whole_cube.im3    25 MB  z -1.43 .. +9.09


### Check — Part B


In [16]:
# =============================================================================
# Part B, check: the same round trip as Part A, then a last look at what the
# trim actually removed, leg by leg.
# =============================================================================
L.verify_norm(concat_root(TRIMMED))

for src_path in sorted(concat_raw(TRIMMED).glob("*_concat.im3")):
    side = json.loads(src_path.with_name(src_path.stem + "_channels.json").read_text())
    dropped = 217 - side["C"]
    blocks = "  ".join(f"{s['excitation']}:{s['channels']}" for s in side["sources"])
    print(f"{side['leg']:24s} {side['C']} channels ({dropped} dropped)   {blocks}")


way 2 - z-norm per channel
  12.09_Rat_1_Left_leg_concat_znorm_per_channel.im3    C=210 scopes=210  z -4.11..+75.28  step 0.0012 sigma  round trip <= 0.52 counts  u 338-26032
  12.09_Rat_1_Right_leg_concat_znorm_per_channel.im3   C=210 scopes=210  z -4.81..+29.20  step 0.0005 sigma  round trip <= 0.17 counts  u 1165-25593
  12.09_Rat_2_Left_leg_concat_znorm_per_channel.im3    C=210 scopes=210  z -2.96..+22.00  step 0.0004 sigma  round trip <= 0.14 counts  u 943-25488
  12.09_Rat_2_Right_leg_concat_znorm_per_channel.im3   C=210 scopes=210  z -4.43..+29.64  step 0.0005 sigma  round trip <= 0.17 counts  u 613-26797
  12.09_Rat_3_Left_leg_concat_znorm_per_channel.im3    C=210 scopes=210  z -3.30..+43.30  step 0.0007 sigma  round trip <= 0.22 counts  u 366-15014
  12.09_Rat_3_Right_leg_concat_znorm_per_channel.im3   C=210 scopes=210  z -2.58..+49.68  step 0.0008 sigma  round trip <= 0.21 counts  u 238-22931

way 3 - z-norm per excitation cube
  12.09_Rat_1_Left_leg_concat_znorm_per_excitati

# What was built


In [17]:
# =============================================================================
# The finished trees: one cube per leg per way, one mask per leg per tree.
# =============================================================================
for tree, n_ch in ((ALL_KEPT, 217), (TRIMMED, 210)):
    print(f"\n{tree.name}  ({n_ch} channels)")
    for folder in ["Raw_Concatenated"] + [w["folder"] for w in L.WAYS.values()]:
        d = concat_root(tree) / folder
        cubes = sorted(d.glob("*.im3"))
        size = sum(p.stat().st_size for p in cubes) / 1e9
        print(f"    Concated_cubes/{folder:32s} {len(cubes)} cubes  {size:5.2f} GB  "
              f"+ {len(list(d.glob('*_channels.*')))} sidecars")
    pngs = sorted(concat_labels(tree).glob("*.png"))
    print(f"    Concated_cubes_labels/{'':26s} {len(pngs)} masks + classes.json, labels.json")
    legdirs = [p for p in distinct(tree).iterdir() if p.is_dir()]
    n = sum(len(list(p.glob('*.im3'))) for p in legdirs)
    print(f"    Distinct_cubes/{'':32s} {len(legdirs)} legs, {n} cubes")
    print(f"    Distinct_cubes_labels/{'':26s} "
          f"{sum(len(list(p.glob('*.png'))) for p in distinct_labels(tree).iterdir() if p.is_dir())} masks")



Alligned_Cropped_All_channels_kept  (217 channels)
    Concated_cubes/Raw_Concatenated                 6 cubes   0.26 GB  + 12 sidecars
    Concated_cubes/Per_Chanell_Norm_1_99_Concat     6 cubes   0.26 GB  + 12 sidecars
    Concated_cubes/Per_Cube_Norm_1_99_Concat        6 cubes   0.26 GB  + 12 sidecars
    Concated_cubes/Overall_4D_Norm_1_99_Concat      6 cubes   0.26 GB  + 12 sidecars
    Concated_cubes_labels/                           6 masks + classes.json, labels.json
    Distinct_cubes/                                 6 legs, 42 cubes
    Distinct_cubes_labels/                           42 masks

Alligned_cropped_overexposed_channels_removed  (210 channels)
    Concated_cubes/Raw_Concatenated                 6 cubes   0.25 GB  + 12 sidecars
    Concated_cubes/Per_Chanell_Norm_1_99_Concat     6 cubes   0.25 GB  + 12 sidecars
    Concated_cubes/Per_Cube_Norm_1_99_Concat        6 cubes   0.25 GB  + 12 sidecars
    Concated_cubes/Overall_4D_Norm_1_99_Concat      6 cubes   0.25 GB 

---

**Reading a leg back for training** — the cube and its mask, on the same grid:

```python
import sys, numpy as np; sys.path.insert(0, "Code")
import concat_lib as L
from PIL import Image

tree = L.PROCESSED_ROOT / "Alligned_cropped_overexposed_channels_removed"
cube = tree / "Concated_cubes/Per_Cube_Norm_1_99_Concat/12.09_Rat_1_Left_leg_concat_znorm_per_excitation.im3"
mask = np.array(Image.open(tree / "Concated_cubes_labels/12.09_Rat_1_Left_leg_concat.png"))

info = L.read_cube_info(cube)
X = np.memmap(cube, "<u2", "r", offset=info["data_offset"], shape=(info["Y"], info["X"], info["C"]))
assert mask.shape == X.shape[:2]
# X[y, x] is that pixel's 210-value spectrum; mask[y, x] its class id (255 = unlabelled).
```

The pixels are `uint16`; `<cube>_channels.json` beside it carries `z_min`, `z_max` and
the per-channel `mean`/`std` that invert them back to counts. **Split train/val by
leg**, never by pixel — one mask covers a whole leg, and neighbouring pixels are not
independent samples.
